# 02 — Explore Duplicates (products sold by multiple shops)

Using the product cache built by `01_explore_catalog.ipynb`, check how many real
products are genuinely **duplicated across independent shops** — i.e. have ≥2 offers
from different merchant domains — and what currencies/merchants actually show up.

UPCitemdb's `search` results already embed an `offers` array per item. We use that
first (zero extra API cost). For items with no embedded offers, we optionally backfill
with a targeted `lookup` call per UPC — budget-capped and sharing the same 100/day
quota as notebook 01.

In [1]:
import json
import time
from datetime import date
from pathlib import Path

import pandas as pd
import requests


In [2]:
# --- config (mirrors notebook 01) -------------------------------------------

CACHE_DIR = Path("../data/catalog")
PRODUCTS_CACHE = CACHE_DIR / "products.jsonl"
OFFERS_CACHE = CACHE_DIR / "offers_lookup.jsonl"   # upc -> offers, from targeted lookup backfill
QUOTA_FILE = CACHE_DIR / "api_quota.json"          # shared with notebook 01

BASE_URL = "https://api.upcitemdb.com/prod/trial"
HEADERS = {"Content-Type": "application/json", "Accept": "application/json"}

DAILY_QUOTA = 100
SAFETY_BUFFER = 10
MAX_LOOKUPS_THIS_RUN = 15   # backfill budget for this run — most items already have embedded offers


In [3]:
def _load_quota():
    if QUOTA_FILE.exists():
        state = json.loads(QUOTA_FILE.read_text())
        if state.get("date") == str(date.today()):
            return state
    return {"date": str(date.today()), "used": 0}


def quota_remaining():
    return DAILY_QUOTA - _load_quota()["used"]


def record_request(n=1):
    state = _load_quota()
    state["used"] += n
    QUOTA_FILE.write_text(json.dumps(state))
    return DAILY_QUOTA - state["used"]


print(f"Requests remaining today (shared with notebook 01): {quota_remaining()}")


Requests remaining today (shared with notebook 01): 0


In [4]:
def load_jsonl(path):
    if not path.exists():
        return []
    out = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out


products = load_jsonl(PRODUCTS_CACHE)
print(f"Loaded {len(products)} cached products from notebook 01")

df = pd.json_normalize(products)
df.head()


Loaded 184 cached products from notebook 01


,ean,title,description,upc,brand,model,color,size,dimension,weight,category,currency,lowest_recorded_price,highest_recorded_price,images,offers,asin,elid
0,0610214633491,LG Optimus F6 4GB Android 4G LTE Smartphone - ...,Time to talk. Upgrade your phone and enjoy you...,610214633491,LG Electronics,D500,Black,4GB,,,Electronics > Communications > Telephony > Mob...,,1.99,248.83,[http://images.qvc.com/is/image/e/63/e226363.0...,"[{'merchant': 'Newegg.com', 'domain': 'newegg....",B00HRJI0EU,202392643777
1,0681131159623,"Onn Virtual Reality Headset, White","ONN Virtual Reality Headset, White:Universal f...",681131159623,onn.,0068113115962,White,,,,Electronics > Audio > Audio Components > Headp...,,1.00,70.00,[https://i5.walmartimages.com/asr/d4293a92-e8d...,"[{'merchant': 'Newegg.com', 'domain': 'newegg....",NaN,326233569476
2,0703558783806,"Indigi P1 Pro Android Pie SmartPhone, GSM Unlo...","Indigi P1 Pro Android Pie SmartPhone, GSM Unlo...",703558783806,Indigi,SMARTPHONE-P1-BK-02,Black,,,,Electronics > Communications > Telephony > Mob...,,129.99,349.99,[https://i5.walmartimages.com/asr/bcba151f-827...,"[{'merchant': 'Rakuten(Buy.com)', 'domain': 'r...",NaN,133442876639
3,0703558783813,"Indigi® P1 Gaming SmartPhone, 6.3-inch HD Disp...","Indigi P1 Gaming SmartPhone, 6.3-inch HD Displ...",703558783813,Indigi,SMARTPHONE-P1-BK-03,Black,,,,Electronics,,129.99,129.99,[https://i5.walmartimages.com/asr/324c59e4-529...,"[{'merchant': 'Rakuten(Buy.com)', 'domain': 'r...",NaN,143368964187
4,0703558783905,"Indigi P1 Pro Android SmartPhone, GSM Unlocked...","Indigi P1 Pro Android SmartPhone, GSM Unlocked...",703558783905,Indigi,SMARTPHONE-P1-RED-02,Red,,,,Electronics > Communications > Telephony > Mob...,,129.99,129.99,[https://i5.walmartimages.com/asr/019f54ba-f1e...,"[{'merchant': 'Rakuten(Buy.com)', 'domain': 'r...",NaN,401870874263


In [5]:
# how many already have offers embedded directly from the search response?
def offer_list(item):
    o = item.get("offers")
    return o if isinstance(o, list) else []

embedded_counts = pd.Series([len(offer_list(p)) for p in products])
print("Embedded-offer count distribution (from search results, no extra API cost):")
print(embedded_counts.describe())
print()
print(f"Products with 0 embedded offers: {(embedded_counts == 0).sum()} / {len(products)}")


Embedded-offer count distribution (from search results, no extra API cost):
count    184.000000
mean       7.173913
std        6.875328
min        0.000000
25%        2.000000
50%        4.000000
75%       11.000000
max       29.000000
dtype: float64

Products with 0 embedded offers: 16 / 184


In [6]:
# --- optional backfill: targeted lookup for items with 0 embedded offers ----
# Same burst-limit handling as script 01's search call: wait out 429s and retry the
# same request instead of abandoning it, and don't count 429s against daily quota.
# A Retry-After above MAX_BURST_WAIT_SECONDS means the daily quota is actually
# exhausted (confirmed live: server reports hours, not seconds) — stop instead of
# sleeping through it.

MAX_RETRIES_PER_REQUEST = 5
SLEEP_BETWEEN_REQUESTS = 1.0
MAX_BURST_WAIT_SECONDS = 120  # a Retry-After bigger than this is the daily reset, not a burst cooldown


def lookup_upcitemdb(upc):
    params = {"upc": upc}

    for attempt in range(MAX_RETRIES_PER_REQUEST):
        try:
            resp = requests.get(f"{BASE_URL}/lookup", headers=HEADERS, params=params, timeout=30)
        except requests.exceptions.RequestException as e:
            print(f"Network error ({e}); retrying in 3s...")
            time.sleep(3)
            continue

        if resp.status_code == 429:
            retry_after = float(resp.headers.get("Retry-After", 5))
            if retry_after > MAX_BURST_WAIT_SECONDS:
                state = _load_quota()
                state["used"] = DAILY_QUOTA
                QUOTA_FILE.write_text(json.dumps(state))
                raise RuntimeError(
                    f"Daily quota actually exhausted (server says wait {retry_after/3600:.1f}h). "
                    "Stopping now — resume after the daily reset."
                )
            print(f"Burst limit hit, waiting {retry_after:.1f}s (not counted against daily quota)...")
            time.sleep(retry_after + 0.5)
            continue

        record_request(1)

        if resp.status_code == 404:
            time.sleep(SLEEP_BETWEEN_REQUESTS)
            return {"items": []}

        resp.raise_for_status()
        time.sleep(SLEEP_BETWEEN_REQUESTS)
        return resp.json()

    raise RuntimeError("Repeated burst-limit/network failures after retries — stopping this run, resume later.")


In [7]:
# --- merge embedded + backfilled offers into one working table --------------

backfilled = {o["upc"]: o["offers"] for o in load_jsonl(OFFERS_CACHE) if o.get("upc")}

rows = []
for p in products:
    upc = p.get("upc")
    offers = offer_list(p)
    if not offers and upc in backfilled:
        offers = backfilled[upc]

    merchants = {o.get("domain") or o.get("merchant") for o in offers if o.get("domain") or o.get("merchant")}
    prices = [o["price"] for o in offers if isinstance(o.get("price"), (int, float)) and o["price"] > 0]
    currencies = {o.get("currency") for o in offers if o.get("currency")}

    rows.append({
        "upc": upc,
        "title": p.get("title"),
        "brand": p.get("brand"),
        "category_top": (p.get("category") or "").split(">")[0].strip(),
        "offer_count": len(offers),
        "distinct_merchant_count": len(merchants),
        "is_duplicated_across_shops": len(merchants) >= 2,
        "min_price": min(prices) if prices else None,
        "max_price": max(prices) if prices else None,
        "currencies_seen": sorted(c for c in currencies if c),
    })

summary = pd.DataFrame(rows)
summary.head()


,upc,title,brand,category_top,offer_count,distinct_merchant_count,is_duplicated_across_shops,min_price,max_price,currencies_seen
0,610214633491,LG Optimus F6 4GB Android 4G LTE Smartphone - ...,LG Electronics,Electronics,4,4,True,119.99,248.83,[CAD]
1,681131159623,"Onn Virtual Reality Headset, White",onn.,Electronics,2,2,True,10.99,11.19,[]
2,703558783806,"Indigi P1 Pro Android Pie SmartPhone, GSM Unlo...",Indigi,Electronics,2,2,True,129.99,129.99,[]
3,703558783813,"Indigi® P1 Gaming SmartPhone, 6.3-inch HD Disp...",Indigi,Electronics,2,2,True,129.99,129.99,[]
4,703558783905,"Indigi P1 Pro Android SmartPhone, GSM Unlocked...",Indigi,Electronics,2,2,True,129.99,129.99,[]


## The question we actually came here to answer

In [8]:
dup_rate = summary["is_duplicated_across_shops"].mean()
print(f"Products sold by >=2 distinct shops: {summary['is_duplicated_across_shops'].sum()} / {len(summary)} ({dup_rate:.1%})")
print()
print("Distribution of distinct-merchant counts:")
print(summary["distinct_merchant_count"].value_counts().sort_index())


Products sold by >=2 distinct shops: 156 / 184 (84.8%)

Distribution of distinct-merchant counts:
distinct_merchant_count
0     16
1     12
2     36
3     26
4      3
5     17
6      1
7      4
8      9
9      6
10     7
11     7
12     7
13     5
14     4
15     2
16     2
17     2
18     4
19     3
21     2
23     2
25     2
26     1
27     2
28     1
29     1
Name: count, dtype: int64


In [9]:
# currency reality check — how much genuine EUR coverage do we actually have?
all_currencies = pd.Series([c for cs in summary["currencies_seen"] for c in cs])
print("Currency distribution across all offers:")
print(all_currencies.value_counts())


Currency distribution across all offers:
CAD    83
GBP     5
EUR     1
Name: count, dtype: int64


In [10]:
# top merchants among duplicated products, and by category
dup_only = summary[summary["is_duplicated_across_shops"]]
print(f"Duplicated-product category breakdown ({len(dup_only)} products):")
print(dup_only["category_top"].value_counts())


Duplicated-product category breakdown (156 products):
category_top
Electronics              135
Apparel & Accessories      7
Cameras & Optics           5
Hardware                   4
                           2
Home & Garden              1
Office Supplies            1
Sporting Goods             1
Name: count, dtype: int64


In [11]:
# persist the summary for later use when we actually build the pipeline
out_path = CACHE_DIR / "duplicate_summary.csv"
summary.to_csv(out_path, index=False)
print(f"Saved summary for {len(summary)} products to {out_path}")


Saved summary for 184 products to ../data/catalog/duplicate_summary.csv


**Read on the result (confirmed via smoke test, not just a guess):** EUR offers do
exist (e.g. Argos Ireland), but they're rare — one EUR offer out of dozens reviewed, and
always a *single* EUR merchant on a product, never two independent EUR shops on the same
item at once. So real multi-shop EUR price duplication isn't something this data actually
gives us — and UPCitemdb's offers lean US/Canada-heavy anyway. Decision: compete in
**USD**, not EUR — matches the data instead of fighting it. Keep *product identity*
(title/brand/model/category) real — that's the part UPCitemdb genuinely delivers — and
treat the *shop-level USD pricing* (our fictional competitor shops, their prices) as
synthetic, generated on top of the real catalog rather than scraped.